# 03 — Análise do primeiro treino: CycleGAN completa x sem *cycle loss*

Este notebook não tem `TODO`s. Ele é uma **análise guiada** do primeiro treino de verdade (a noite de 2026-10-29), para você entender uma GAN na prática: o que os números significam, como ler as curvas, onde o modelo errou e por quê.

Cada seção lê os arquivos gerados pelo treino (nada é digitado à mão) e alguns testes rodam o modelo na hora. Mude os números, rode de novo, experimente.

O resumo com todos os números está no registro do experimento: `docs/experimentos/2026-10-29-primeiro-treino.md`.

| Seção | Pergunta |
|---|---|
| 1 | O que exatamente foi rodado? |
| 2 | Quanto tempo levou, e a máquina aguentou? |
| 3 | O que as curvas de loss contam? |
| 4 | Como as imagens evoluíram? |
| 5 | O que as métricas dizem, e onde elas enganam? |
| 6 | Onde a GAN mais errou? |
| 7 | Ela copiou as pinturas? |
| 8 | Duas hipóteses testadas na prática |
| 9 | O objetivo do projeto: o céu de Van Gogh |
| 10 | As previsões bateram? |
| 11 | O que mudar no próximo treino |

In [ ]:
import sys
sys.path.insert(0, "..")

import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from src.models import Generator
from src.utils import OUTPUTS_DIR, ROOT, get_device, load_domain, load_image, to_float, to_unit

device = get_device()
FIG = ROOT / "report" / "figuras"
RUN = OUTPUTS_DIR / "primeiro_treino"
MODELOS = {"completa": "cyclegan", "sem cycle": "cyclegan_sem_cycle"}


def mostra_arquivo(caminho, titulo=None, largura=16):
    # Mostra um PNG salvo pelo treino.
    img = plt.imread(caminho)
    plt.figure(figsize=(largura, largura * img.shape[0] / img.shape[1]))
    plt.imshow(img)
    plt.axis("off")
    if titulo:
        plt.title(titulo)
    plt.show()


def mostra(imagens, titulo=None, nrow=8):
    # Mostra um tensor (N, 3, H, W) em [-1, 1].
    from torchvision.utils import make_grid
    grade = make_grid(to_unit(imagens.detach().float().cpu()).clamp(0, 1), nrow=nrow, padding=2)
    plt.figure(figsize=(2 * nrow, 2 * ((len(imagens) + nrow - 1) // nrow)))
    plt.imshow(grade.permute(1, 2, 0))
    plt.axis("off")
    if titulo:
        plt.title(titulo)
    plt.show()


def carrega_gerador(modelo, epoca=40, qual="G"):
    ck = torch.load(OUTPUTS_DIR / modelo / "checkpoints" / f"epoch_{epoca:03d}.pt", map_location=device)
    net = Generator(n_blocks=ck["args"]["n_blocks"]).to(device).eval()
    net.load_state_dict(ck[qual])
    return net

---
# 1. O que foi rodado

Cada checkpoint guarda os argumentos do treino em `args`. Em vez de confiar na memória ou na documentação, lemos direto de lá: é a fonte mais confiável do que realmente rodou.

In [ ]:
config = {}
for nome, pasta in MODELOS.items():
    ck = torch.load(OUTPUTS_DIR / pasta / "checkpoints" / "epoch_040.pt", map_location="cpu")
    config[nome] = ck["args"]
pd.DataFrame(config)

**Repare:** as duas colunas só diferem em `lambda_cycle` (10 x 0), `sem_cycle` e `out`. Mesma seed, mesmas 2.000 fotos, mesmas épocas. É isso que torna a comparação justa: uma única coisa muda, e qualquer diferença no resultado vem dela.

Isso se chama **estudo de ablação** (*ablation study*): tirar uma peça do modelo para medir o quanto ela contribui. É um formato muito comum em artigos de aprendizado de máquina.

---
# 2. Tempo e hardware

O script da noite registrou o tempo de cada passo (`resumo.json`) e o estado da GPU a cada minuto (`gpu.csv`).

In [ ]:
resumo = json.loads((RUN / "resumo.json").read_text(encoding="utf-8"))
display(pd.DataFrame(resumo["passos"]))

gpu = pd.read_csv(RUN / "gpu.csv", skipinitialspace=True)
gpu.columns = [c.split(" ")[0] for c in gpu.columns]
for col in ("temperature.gpu", "clocks.sm", "power.draw", "utilization.gpu"):
    gpu[col] = pd.to_numeric(gpu[col].astype(str).str.extract(r"([\d.]+)")[0])
gpu["minuto"] = range(len(gpu))

fig, eixos = plt.subplots(1, 3, figsize=(15, 3))
for ax, (col, titulo) in zip(eixos, [("temperature.gpu", "temperatura (°C)"),
                                     ("clocks.sm", "clock (MHz)"),
                                     ("utilization.gpu", "uso da GPU (%)")]):
    ax.plot(gpu["minuto"], gpu[col])
    ax.set_title(titulo)
    ax.set_xlabel("minuto desde o início")
plt.tight_layout()
plt.show()

**Como ler:**
- A GPU ficou entre 70 e 76 °C e perto do clock máximo (~1.912 de 2.100 MHz) a noite toda: **não houve superaquecimento** nem redução de clock.
- A **1ª época** do treino completo levou 8,7 min, e as demais, 4,2 min. A primeira é mais lenta porque o `cudnn.benchmark` testa os algoritmos de convolução e os dados são carregados pela primeira vez.
- **Curiosidade:** sem a *cycle loss*, o gerador roda 2 vezes a menos por passo (não precisa calcular a volta), e mesmo assim a época levou os mesmos 4,2 min. Isso indica que o gargalo não é só a conta na GPU: o Python, a cópia dos batches para a GPU e o *image pool* também pesam. É uma pista para acelerar os próximos treinos.

---
# 3. As curvas de loss

O `logs.csv` tem uma linha a cada 100 passos com as losses daquele passo. Como o batch é 1, um passo isolado oscila muito. Por isso olhamos a **média por época**.

In [ ]:
logs = {nome: pd.read_csv(OUTPUTS_DIR / pasta / "logs.csv") for nome, pasta in MODELOS.items()}
medias = {nome: df.groupby("epoch").mean(numeric_only=True) for nome, df in logs.items()}

fig, eixos = plt.subplots(2, 3, figsize=(16, 8), sharex=True)
paineis = [("loss_cycle", "cycle (ida e volta)"), ("loss_identity", "identity"),
           ("loss_g_adv", "G adversarial (foto → pintura)"), ("loss_f_adv", "F adversarial (pintura → foto)"),
           ("loss_d_pintura", "D_pintura"), ("loss_d_foto", "D_foto")]
for ax, (col, titulo) in zip(eixos.flat, paineis):
    for nome, m in medias.items():
        ax.plot(m.index, m[col], label=nome, marker=".")
    if col.startswith("loss_d"):
        ax.axhline(0.25, ls="--", c="gray", label="D chutando (0.25)")
    ax.set_title(titulo)
    ax.legend(fontsize=8)
for ax in eixos[1]:
    ax.set_xlabel("época")
plt.tight_layout()
plt.show()

### O que cada curva conta

**`cycle`:** na completa, ela cai de ~0,6 para ~0,2: o modelo aprende a ir e voltar sem perder a foto. Na sem *cycle*, ela é só **medida**, não otimizada, e **sobe** a partir da época ~25, quando o modelo colapsa. Repare que é a única curva que mostra claramente o problema.

**`identity`:** cai nos dois treinos, porque ela continua ligada nos dois. No começo, ela também segura o conteúdo da foto no treino sem *cycle*, mas não é suficiente (seção 4).

**Discriminadores:** abaixo de 0,25 o D está acertando mais que um chute. Nos dois treinos, o **D_pintura** termina bem baixo (~0,03 a 0,07): o discriminador de pinturas ganhou o jogo. Com só 400 pinturas, vistas ~5 vezes por época, fica fácil decorar quais são reais.

**`G adversarial`:** sobe na segunda metade da completa (de ~0,5 para ~0,9). É o outro lado da mesma moeda: se o D_pintura fica mais forte, enganá-lo fica mais difícil.

### A lição mais importante desta seção

No treino sem *cycle*, a G_adv e a D_pintura parecem **normais**: nada nelas grita "colapsou". Numa GAN, **as losses adversariais não medem qualidade**. Elas medem quem está ganhando o jogo naquele momento, e o jogo pode estar "equilibrado" com o gerador produzindo lixo. Por isso olhamos imagens e métricas separadas.

**Pergunta:** se você tivesse só a curva da G_adv do treino sem *cycle*, conseguiria dizer em que época o modelo colapsou? Que curva ou métrica você usaria para criar um "alarme de colapso" automático?

✍️ **Sua resposta:** _dê dois cliques nesta célula para escrever_

---
# 4. A evolução das imagens

A cada época, o treino salva uma grade com as **mesmas 8 fotos de teste**. Linha 1 = foto, linha 2 = G(foto) (a "pintura"), linha 3 = F(G(foto)) (a volta). Mude a lista `epocas` para ver outras.

In [ ]:
epocas = [1, 10, 20, 40]
for nome, pasta in MODELOS.items():
    for e in epocas:
        mostra_arquivo(OUTPUTS_DIR / pasta / "samples" / f"epoch_{e:03d}.png", f"{nome}: época {e}", largura=14)

### O que aconteceu, fase por fase

**Completa:**
- **Época 1:** cores já puxando para o amarelo e o verde, mas sem pincelada.
- **Época 10:** o estilo apareceu (paleta, textura de pincel, céus com movimento). A volta já reconstrói bem.
- **Época 20:** primeiros artefatos em céus lisos (listras e padrão de grade na casinha).
- **Época 40:** pinceladas mais longas e marcadas. Problemas que ficaram: o templo esverdeado (o gerador puxa a paleta inteira para os tons de Van Gogh) e o pato branco apagado.

**Sem *cycle loss*:**
- **Época 5:** surpresa: o conteúdo **foi preservado** e o estilo, fraco. No começo, a *identity loss* e a arquitetura residual (que facilita "não mudar nada") seguram a foto.
- **Época 10 a 20:** o gerador passa a **inventar** (a praia vira mato, os patos mudam de cor) e o estilo fica até **mais forte** que o da completa.
- **Época 25 a 40:** **colapso de modo**. As fotos ganham a mesma "assinatura" e, no fim, 7 das 8 viram praticamente a mesma imagem.

Abra também `outputs/cyclegan/evolucao.gif` e `outputs/cyclegan_sem_cycle/evolucao.gif` para ver as 40 épocas em sequência.

---
# 5. As métricas e onde elas enganam

A avaliação usa as **751 fotos de teste**, que o modelo nunca viu, e compara as "pinturas" geradas com as **400 pinturas reais**.

In [ ]:
metricas = {nome: json.loads((OUTPUTS_DIR / pasta / "metrics.json").read_text(encoding="utf-8"))
            for nome, pasta in MODELOS.items()}
campos = ["fid", "fid_fotos_originais", "kid", "kid_std", "diversidade", "diversidade_pinturas",
          "cycle_l1", "mudanca_l1", "nn_dist_fake_mean", "nn_dist_ref_mean", "nn_dist_fake_min"]
pd.DataFrame(metricas).loc[campos].astype(float).round(4)

### Lendo a tabela

- **FID:** a foto sem conversão dá 179,8 (é a "linha de base"). A completa leva para **125,3**: as saídas ficaram bem mais próximas das pinturas. A sem *cycle* dá **220,9**, ou seja, **pior que não fazer nada**.
- **KID:** mesma história (0,051 x 0,190, com base de 0,100), mas com desvio padrão, que mostra que a diferença é real e não ruído.
- **Diversidade:** distância média entre todas as saídas. As pinturas reais dão 137,7, e a completa, 128,9 (perto). A sem *cycle* dá **56,9**: as saídas ficaram parecidas demais entre si. É o colapso em forma de número.
- **`cycle_l1`:** 0,130 x 0,494. A completa preserva a foto, a sem *cycle* não.

### Uma previsão que errou

Durante a noite, ao ver a grade da época 20, a previsão registrada foi: *"o FID da versão sem cycle pode sair igual ou melhor que o da completa"*, porque ela parecia "mais Van Gogh". Errou no fim. Mas a tabela por checkpoint mostra que ela **acertava até a época 15**:

In [ ]:
por_ck = pd.read_csv(RUN / "por_checkpoint.csv")
fig, eixos = plt.subplots(1, 3, figsize=(16, 3.8))
for ax, col, titulo in zip(eixos, ["fid", "diversidade", "cycle_l1"],
                           ["FID (menor = melhor)", "diversidade das saídas", "erro de ida e volta"]):
    for pasta, df in por_ck.groupby("modelo"):
        ax.plot(df["epoca"], df[col], marker="o", label=pasta)
    ax.set_title(titulo)
    ax.set_xlabel("época do checkpoint")
eixos[0].axhline(metricas["completa"]["fid_fotos_originais"], ls="--", c="gray", label="foto sem conversão")
eixos[1].axhline(metricas["completa"]["diversidade_pinturas"], ls="--", c="gray", label="pinturas reais")
for ax in eixos:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
por_ck.pivot(index="epoca", columns="modelo", values=["fid", "diversidade"])

**Por que o FID puniu o colapso?** O FID não pergunta só "cada imagem parece uma pintura?". Ele compara a **distribuição** inteira: a média **e a variação** das features (a covariância). Quando todas as saídas viram a mesma imagem, a variação some, e a distribuição gerada fica muito diferente da distribuição das 400 pinturas, que são variadas. Por isso, a partir da época 20, o FID da sem *cycle* sobe junto com a queda da diversidade.

**E a completa?** O FID caiu bastante até a época 25 (124,2) e depois oscilou entre 125 e 131. Ou seja, **as últimas 15 épocas quase não ajudaram**. Esse é o tipo de informação que só aparece avaliando vários checkpoints, e não só o último.

**O que nenhuma dessas métricas mede:** se o céu ficou bonito, se a pincelada parece a de Van Gogh e não a de outro pintor, se a cena faz sentido. Para isso, olhar as imagens continua indispensável.

> Para recalcular a tabela por checkpoint (cerca de 5 min), rode `src/evaluate.py` com `--checkpoint epoch_NNN.pt` para cada época, ou adapte o código das funções `translate`, `style_metrics` e `diversidade` do `src/evaluate.py`.

**Pergunta:** se você só pudesse reportar **um** número no artigo para comparar os dois modelos, qual escolheria? E por que reportar só um seria perigoso?

✍️ **Sua resposta:** _dê dois cliques nesta célula para escrever_

---
# 6. Onde a GAN mais errou

O `evaluate.py` separa as 8 fotos de teste com **maior erro de ida e volta**: aquelas em que o modelo mais "perdeu" a foto. Cada linha: foto | pintura | volta.

In [ ]:
mostra_arquivo(FIG / "cyclegan_piores.png", "completa: os 8 piores casos", largura=6)

### O padrão dos piores casos

A maioria envolve **céus difíceis**:
- **Céus noturnos:** a Via Láctea (linha 1) e um céu azul-escuro estrelado (linha 2). O modelo só viu paisagens **diurnas**: uma foto escura está fora da distribuição de treino, e ele pinta por cima com amarelos de dia.
- **Céus lisos e saturados de pôr do sol:** o sol vermelho (linha 5), o céu cor-de-rosa (linhas 4 e 6).

**Por que isso importa tanto:** o objetivo do projeto é pintar **céus**, e a obra mais famosa que inspirou a ideia é uma **noite** estrelada. O modelo vai pior justamente no caso mais desejado. Isso vira uma limitação a declarar no artigo e uma decisão a tomar: deixar a noite fora do escopo ou incluir fotos noturnas no treino.

### Os outros defeitos vistos nas grades

| Defeito | Onde | Causa provável |
|---|---|---|
| Listras e padrão de grade em céus lisos | céu da casinha | convoluções transpostas do decodificador (*checkerboard artifacts*) |
| Cores trocadas (templo esverdeado) | regiões cinzas ou neutras | o gerador puxa tudo para a paleta de Van Gogh |
| Detalhe pequeno e claro some (pato branco) | objetos pequenos de baixo contraste | 128x128 tem pouca resolução, e o branco se mistura à textura |

---
# 7. A GAN copiou as pinturas?

Para cada saída, procuramos a pintura real mais parecida (distância L2 em pixels). A referência é o quanto **duas pinturas reais diferentes** se parecem entre si.

In [ ]:
mostra_arquivo(FIG / "cyclegan_memorizacao.png", "distâncias até a pintura mais próxima", largura=8)
mostra_arquivo(FIG / "cyclegan_vizinhos.png", "as 8 saídas mais próximas de uma pintura (saída | pintura)", largura=4)

**Leitura:** as duas distribuições se sobrepõem (média 93,1 x 93,2). As saídas não estão mais perto das pinturas do que uma pintura está de outra. **Não há sinal de cópia.**

A barrinha em **0** no histograma azul são as pinturas que aparecem **duplicadas** no próprio dataset (2 pares, veja `docs/referencia/dados.md`).

**Um detalhe curioso nos vizinhos:** quase todas as saídas têm como "vizinha" a mesma pintura, a *Cadeira de Gauguin*. Ela não foi copiada: ela tem cores médias (verde-escuro, vermelho, marrom) e por isso fica perto do "centro" do espaço de pixels. Em espaços com muitas dimensões, alguns pontos viram vizinhos de quase todo mundo (o fenômeno dos *hubs*). Essa é uma **limitação da distância em pixels**: ela só detecta cópias quase literais, não cópias de composição ou de pedaços.

---
# 8. Duas hipóteses testadas na prática

Durante a noite surgiram duas explicações para o que apareceu nas grades. Aqui elas são testadas com o modelo de verdade.

### Hipótese 1: esteganografia

Na época 15, o pato branco **sumia na pintura mas voltava na reconstrução**. Uma explicação conhecida (Chu et al., 2017) é que a CycleGAN aprende a **esconder** a informação num sinal quase invisível dentro da pintura, só para conseguir reconstruir depois.

**Como testar:** se a informação está escondida num sinal fraco, somar um **ruído pequeno** à pintura deveria destruí-la, e o erro da volta deveria subir muito.

In [ ]:
torch.manual_seed(0)
G, F = carrega_gerador("cyclegan", qual="G"), carrega_gerador("cyclegan", qual="F")
fotos = to_float(load_domain("foto", train=False)).to(device)

resultados = []
with torch.no_grad():
    pinturas_geradas = torch.cat([G(fotos[i:i + 16]) for i in range(0, len(fotos), 16)])
    for sigma in (0.0, 0.02, 0.05, 0.1, 0.2):
        ruidosas = (pinturas_geradas + sigma * torch.randn_like(pinturas_geradas)).clamp(-1, 1)
        volta = torch.cat([F(ruidosas[i:i + 16]) for i in range(0, len(fotos), 16)])
        resultados.append({"ruído (σ)": sigma, "erro da volta": (volta - fotos).abs().mean().item()})
pd.DataFrame(resultados).round(3)

**Resultado:** o erro da volta quase não muda com ruído pequeno (0,130 → ~0,135 com σ = 0,1). Se o pato dependesse de um sinal escondido e fraco, o ruído o apagaria. **A hipótese não se confirmou** neste modelo: o mais provável é que a informação continue visível na pintura, só que com pouco contraste.

Isso é uma parte importante do método: **uma hipótese bonita registrada no diário não vira conclusão sem teste.**

### Hipótese 2: os artefatos do céu vêm da escala

O `stylize` roda o gerador com o lado menor em 256 pixels, mas o treino foi em 128. A hipótese é que a moldura e a grade no céu liso venham dessa diferença de escala.

In [ ]:
import torch.nn.functional as nnf

arquivos = sorted((ROOT / "data" / "vangogh2photo" / "testB").glob("*.jpg"))
saidas = []
with torch.no_grad():
    for i in (4, 6):   # a casinha (céu liso) e o templo
        x = to_float(load_image(arquivos[i]))[None].to(device)
        for lado in (128, 256):
            xs = nnf.interpolate(x, size=(lado, lado), mode="bilinear", align_corners=False)
            saidas.append(nnf.interpolate(G(xs), size=(256, 256), mode="bilinear", align_corners=False))
mostra(torch.cat(saidas), "esquerda: gerador em 128 | direita: gerador em 256", nrow=2)

**Resultado:** em 128 (a escala do treino), a casinha sai **sem** a moldura e a grade, mas mais borrada, porque o resultado é ampliado depois. Em 256, sai mais detalhada, com os artefatos. **Hipótese confirmada:** um gerador convolucional aceita qualquer tamanho de imagem, mas só "conhece" a escala em que treinou. Para ter detalhe **e** céus limpos, o caminho é treinar em 256.

---
# 9. O objetivo do projeto: o céu de Van Gogh

Até aqui avaliamos o gerador sozinho. O sistema completo junta três peças: o gerador, a máscara do céu (SegFormer pré-treinado) e a composição. Abaixo, três fotos de teste passadas pelo `python -m src.stylize`.

In [ ]:
for painel in sorted((OUTPUTS_DIR / "stylize").glob("*_cyclegan_painel.png")):
    mostra_arquivo(painel, painel.stem, largura=16)

**Leitura:**
- **Templo:** é a ideia da estátua funcionando. A máscara recortou bem o céu, inclusive em volta da torre, e o céu ganhou pinceladas enquanto o templo ficou intacto.
- **Praia:** céu de pôr do sol bem estilizado. A máscara pega só a parte mais alta do céu e deixa o horizonte de fora, então a transição fica visível.
- **Casinha:** o céu liso mostra a moldura e a grade da seção 8, que aparecem mais aqui porque nada na cena "esconde" o artefato.

Ainda falta testar em **fotos suas**, que é o objetivo final: `python -m src.stylize sua_foto.jpg`.

---
# 10. As previsões bateram?

| Previsão | Quando | Resultado |
|---|---|---|
| **Sua:** "o primeiro treino não fica tão bom comparado ao original (o artigo)" | antes do treino | **Confirmada.** O estilo apareceu, mas com menos resolução, pinceladas mais grossas e artefatos em céus lisos |
| **ADR 0002:** sem *cycle loss*, o gerador "inventa" e perde o conteúdo | antes do treino | **Confirmada**, e foi além: o modelo também **colapsou** |
| "O conteúdo foi preservado sem *cycle loss*" (a hipótese da época 5) | durante, 01h35 | **Derrubada** na época 10: a *identity loss* só segurou no começo |
| "FID igual ou melhor sem *cycle loss*" | durante, 02h38 | **Certa até a época 15, errada no fim**, por causa do colapso |
| Esteganografia do pato | durante, 23h29 | **Não confirmada** no teste da seção 8 |
| Artefatos do céu vêm da escala 256 x 128 | depois | **Confirmada** no teste da seção 8 |

---
# 11. O que mudar no próximo treino

Cada sugestão vem de uma observação deste treino:

| # | Mudança | Por quê (evidência) | Custo |
|---|---|---|---|
| 1 | **Recorte aleatório** no aumento de dados (redimensionar para ~143 e recortar 128, como o artigo faz com 286 → 256) | O D_pintura dominou nos dois treinos (seção 3): 400 pinturas são fáceis de decorar. Recortes diferentes da mesma pintura dificultam | Quase nenhum |
| 2 | **Treinar em 256x256** | Pinceladas mais finas, e o `stylize` deixa de rodar fora da escala de treino (seção 8) | ~4 vezes mais tempo por passo |
| 3 | **Upsample + convolução** no lugar das convoluções transpostas | Padrão de grade em céus lisos (seções 6 e 9) | Pequena mudança no `src/models.py` |
| 4 | **25 a 30 épocas** em vez de 40 | O FID parou de melhorar por volta da época 25 (seção 5) | Economiza tempo para os itens 1 e 2 |
| 5 | **Investigar o gargalo de velocidade** | Sem *cycle*, metade das contas e o mesmo tempo (seção 2) | Algumas medições |
| 6 | **Decidir sobre céus noturnos** | Piores casos são noites e céus lisos (seção 6) | Escopo ou dados novos |

**Uma regra de ouro:** mude **uma coisa por vez**, como nesta ablação, ou não dá para saber o que causou a diferença.

**Pergunta final:** com o que você viu aqui, qual das mudanças acima você testaria primeiro, e qual a sua **previsão** para o resultado? Escreva antes de rodar: ela vai para o registro do próximo experimento.

✍️ **Sua resposta:** _dê dois cliques nesta célula para escrever_